# Stress Prediction v10 — ExtraTrees Multi-Seed Ensemble

기존 `combined_final + BP ratio only` 구조를 유지하면서 ExtraTrees의 랜덤 시드를 여러 개 사용해 앙상블합니다.

## 변경점

- 유지: BP Ratio 단독 파생변수
- 유지: Winsorize
- 유지: Gender 제거
- 유지: Feature copy
- 유지: Pair predictor
- 유지: 최근접 이웃 override
- 추가: ExtraTrees seed `42`, `2024`, `777`
- 미세 탐색:
  - Tree quantile: `0.500 ~ 0.540`
  - Pair weight: `0.25 ~ 0.31`
  - Override threshold: `0.14 ~ 0.20`
  - 반올림 여부
- 제출 후보 최대 5개 생성

> 계산량이 크면 `N_ESTIMATORS_PER_SEED=600` 또는 `SEEDS=[42, 2024]`로 줄여도 됩니다.


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"

RANDOM_SEED = 42
N_SPLITS = 5

SEEDS = [42, 2024, 777]
N_ESTIMATORS_PER_SEED = 800

TREE_QUANTILES = np.round(
    np.arange(0.500, 0.5401, 0.005),
    3,
).tolist()

PAIR_QUANTILE = 0.48
PAIR_WEIGHTS = np.round(
    np.arange(0.25, 0.3101, 0.01),
    2,
).tolist()

OVERRIDE_THRESHOLDS = [
    None,
    0.14,
    0.15,
    0.16,
    0.17,
    0.18,
    0.19,
    0.20,
]

ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("SEEDS:", SEEDS)
print("Trees per seed:", N_ESTIMATORS_PER_SEED)
print("Total trees:", len(SEEDS) * N_ESTIMATORS_PER_SEED)
print("Tree quantiles:", TREE_QUANTILES)
print("Pair weights:", PAIR_WEIGHTS)


SEEDS: [42, 2024, 777]
Trees per seed: 800
Total trees: 2400
Tree quantiles: [0.5, 0.505, 0.51, 0.515, 0.52, 0.525, 0.53, 0.535, 0.54]
Pair weights: [0.25, 0.26, 0.27, 0.28, 0.29, 0.3, 0.31]


## 1. 데이터 로드


In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

y = train_raw[TARGET].reset_index(drop=True)

raw_input = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)

test_input = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_cols = raw_input.select_dtypes(
    include=["number"]
).columns.tolist()

print("Train:", train_raw.shape)
print("Test :", test_raw.shape)
display(train_raw.head())


Train: (3000, 18)
Test : (3000, 17)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 2. 기존 전처리·피처·Pair 함수


In [3]:
def compute_iqr_bounds(
    train_frame: pd.DataFrame,
    columns: list[str],
):
    bounds = {}

    for col in columns:
        q1, q3 = train_frame[col].quantile(
            [0.25, 0.75]
        )
        iqr = q3 - q1

        bounds[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr,
        )

    return bounds


def apply_winsorize(
    frame: pd.DataFrame,
    bounds,
):
    result = frame.copy()

    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(
            lower,
            upper,
        )

    return result


def add_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = (
        result["weight"]
        / (height_m.pow(2) + 1e-6)
    )

    # A/B 테스트에서 가장 안정적이었던 혈압비만 유지
    result["blood_pressure_ratio"] = (
        sbp / (dbp + 1e-6)
    )

    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"]
        / (result["glucose"] + 1e-6)
    )

    result["missing_count"] = (
        result.isna().sum(axis=1)
    )

    return result


def add_feature_copies(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue

        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = (
                result[feature]
            )

    return result


def make_preprocessor(
    frame: pd.DataFrame,
) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


def extra_trees_seed_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    seed: int,
) -> np.ndarray:
    train_input = train_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    val_input = val_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    train_input = add_feature_copies(
        train_input
    )

    val_input = add_feature_copies(
        val_input
    )

    preprocessor = make_preprocessor(
        train_input
    )

    train_matrix = preprocessor.fit_transform(
        train_input
    )

    val_matrix = preprocessor.transform(
        val_input
    )

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS_PER_SEED,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        train_matrix,
        target,
    )

    return np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])


def empirical_rank_transform(
    train_values,
    test_values,
):
    median = np.nanmedian(train_values)

    train_values = np.nan_to_num(
        train_values,
        nan=median,
    )

    test_values = np.nan_to_num(
        test_values,
        nan=median,
    )

    sorted_train = np.sort(
        train_values
    )

    train_rank = (
        np.searchsorted(
            sorted_train,
            train_values,
            side="right",
        )
        / len(train_values)
    )

    test_rank = (
        np.searchsorted(
            sorted_train,
            test_values,
            side="right",
        )
        / len(train_values)
    )

    return (
        train_rank.astype(np.float32),
        test_rank.astype(np.float32),
    )


def pair_prediction(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    train_rank = {}
    val_rank = {}

    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = (
            empirical_rank_transform(
                train_features[column]
                .to_numpy(dtype=float),
                val_features[column]
                .to_numpy(dtype=float),
            )
        )

    target_values = target.to_numpy(
        dtype=float
    )

    neighbor_targets = []

    for first, second in combinations(
        PAIR_COLUMNS,
        2,
    ):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argmin(
            distance,
            axis=1,
        )

        neighbor_targets.append(
            target_values[nearest_idx]
        )

    return np.quantile(
        np.column_stack(neighbor_targets),
        PAIR_QUANTILE,
        axis=1,
    )


def nearest_neighbor_lookup(
    train_raw_frame: pd.DataFrame,
    val_raw_frame: pd.DataFrame,
    target: pd.Series,
):
    train_numeric = (
        train_raw_frame[numeric_cols]
        .copy()
    )

    val_numeric = (
        val_raw_frame[numeric_cols]
        .copy()
    )

    medians = train_numeric.median()

    train_numeric = train_numeric.fillna(
        medians
    )

    val_numeric = val_numeric.fillna(
        medians
    )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_numeric
    )

    val_scaled = scaler.transform(
        val_numeric
    )

    tree = cKDTree(train_scaled)

    nearest_dist, nearest_idx = tree.query(
        val_scaled,
        k=1,
    )

    nearest_target = (
        target.to_numpy(dtype=float)[nearest_idx]
    )

    return nearest_dist, nearest_target


def apply_postprocess(
    prediction: np.ndarray,
    round_decimals: int | None,
    override_threshold: float | None,
    nearest_dist: np.ndarray,
    nearest_target: np.ndarray,
) -> np.ndarray:
    result = np.clip(
        prediction.copy(),
        0.0,
        1.0,
    )

    if round_decimals is not None:
        result = np.round(
            result,
            round_decimals,
        )

    if override_threshold is not None:
        mask = (
            nearest_dist
            < override_threshold
        )

        replacement = nearest_target[
            mask
        ].copy()

        if round_decimals is not None:
            replacement = np.round(
                replacement,
                round_decimals,
            )

        result[mask] = np.clip(
            replacement,
            0.0,
            1.0,
        )

    return result


## 3. 5-Fold Multi-Seed 캐시 생성

각 Fold에서 동일한 전처리 데이터를 사용해 seed별 ExtraTrees를 학습합니다.  
seed별 모든 tree 예측을 하나로 합친 후 quantile을 계산합니다.


In [4]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

fold_cache = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(raw_input),
    start=1,
):
    train_raw_fold = (
        raw_input.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_raw_fold = (
        raw_input.iloc[val_idx]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    bounds = compute_iqr_bounds(
        train_raw_fold,
        WINSORIZE_COLUMNS,
    )

    train_winsorized = apply_winsorize(
        train_raw_fold,
        bounds,
    )

    val_winsorized = apply_winsorize(
        val_raw_fold,
        bounds,
    )

    train_feat = add_features(
        train_winsorized
    )

    val_feat = add_features(
        val_winsorized
    )

    seed_matrices = []

    for seed in SEEDS:
        seed_matrix = extra_trees_seed_matrix(
            train_feat,
            val_feat,
            y_train,
            seed,
        )

        seed_matrices.append(
            seed_matrix
        )

        print(
            f"Fold {fold}: seed {seed} 완료"
        )

    combined_tree_matrix = np.concatenate(
        seed_matrices,
        axis=1,
    )

    pair_pred = pair_prediction(
        train_feat,
        val_feat,
        y_train,
    )

    nearest_dist, nearest_target = (
        nearest_neighbor_lookup(
            train_raw_fold,
            val_raw_fold,
            y_train,
        )
    )

    fold_cache.append({
        "val_idx": val_idx,
        "tree_matrix": combined_tree_matrix,
        "pair_pred": pair_pred,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })

    print(
        f"Fold {fold}/{N_SPLITS} 캐시 완료"
    )


Fold 1: seed 42 완료
Fold 1: seed 2024 완료
Fold 1: seed 777 완료
Fold 1/5 캐시 완료
Fold 2: seed 42 완료
Fold 2: seed 2024 완료
Fold 2: seed 777 완료
Fold 2/5 캐시 완료
Fold 3: seed 42 완료
Fold 3: seed 2024 완료
Fold 3: seed 777 완료
Fold 3/5 캐시 완료
Fold 4: seed 42 완료
Fold 4: seed 2024 완료
Fold 4: seed 777 완료
Fold 4/5 캐시 완료
Fold 5: seed 42 완료
Fold 5: seed 2024 완료
Fold 5: seed 777 완료
Fold 5/5 캐시 완료


## 4. Multi-Seed 단독 효과 확인


In [5]:
seed_check_rows = []

for tree_q in TREE_QUANTILES:
    oof_pred = np.zeros(
        len(y),
        dtype=float,
    )

    for cache in fold_cache:
        tree_pred = np.quantile(
            cache["tree_matrix"],
            tree_q,
            axis=1,
        )

        oof_pred[
            cache["val_idx"]
        ] = np.round(
            np.clip(
                tree_pred,
                0.0,
                1.0,
            ),
            2,
        )

    mae = mean_absolute_error(
        y,
        oof_pred,
    )

    seed_check_rows.append({
        "tree_quantile": tree_q,
        "tree_only_cv_mae": mae,
    })

seed_check_df = (
    pd.DataFrame(seed_check_rows)
    .sort_values("tree_only_cv_mae")
    .reset_index(drop=True)
)

display(seed_check_df)


,tree_quantile,tree_only_cv_mae
0,0.500,0.147950
1,0.505,0.148000
2,0.510,0.148127
3,0.520,0.148163
4,0.515,0.148183
5,0.525,0.148300
6,0.530,0.148373
7,0.535,0.148407
8,0.540,0.148463


## 5. 미세 Grid Search


In [6]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_weight in PAIR_WEIGHTS:
        for round_decimals in ROUND_OPTIONS:
            for threshold in OVERRIDE_THRESHOLDS:
                oof_pred = np.zeros(
                    len(y),
                    dtype=float,
                )

                override_rows = 0

                for cache in fold_cache:
                    tree_pred = np.quantile(
                        cache["tree_matrix"],
                        tree_q,
                        axis=1,
                    )

                    blended = (
                        (1.0 - pair_weight)
                        * tree_pred
                        + pair_weight
                        * cache["pair_pred"]
                    )

                    final_pred = apply_postprocess(
                        blended,
                        round_decimals,
                        threshold,
                        cache["nearest_dist"],
                        cache["nearest_target"],
                    )

                    oof_pred[
                        cache["val_idx"]
                    ] = final_pred

                    if threshold is not None:
                        override_rows += int(
                            (
                                cache["nearest_dist"]
                                < threshold
                            ).sum()
                        )

                mae = mean_absolute_error(
                    y,
                    oof_pred,
                )

                search_rows.append({
                    "cv_mae": mae,
                    "tree_quantile": tree_q,
                    "pair_weight": pair_weight,
                    "round_decimals": round_decimals,
                    "override_threshold": threshold,
                    "override_rows": override_rows,
                    "seed_count": len(SEEDS),
                    "trees_total": (
                        len(SEEDS)
                        * N_ESTIMATORS_PER_SEED
                    ),
                })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

print("상위 40개 조합")
display(search_df.head(40))

print(
    "최적 OOF MAE:",
    float(search_df.iloc[0]["cv_mae"]),
)


상위 40개 조합


,cv_mae,tree_quantile,pair_weight,round_decimals,override_threshold,override_rows,seed_count,trees_total
0,0.147603,0.54,0.29,2.0,0.16,561,3,2400
1,0.147603,0.54,0.29,2.0,0.19,569,3,2400
2,0.147603,0.54,0.29,2.0,0.20,572,3,2400
3,0.147603,0.54,0.29,2.0,0.18,568,3,2400
4,0.147603,0.54,0.29,2.0,0.14,531,3,2400
5,0.147603,0.54,0.29,2.0,0.15,556,3,2400
6,0.147603,0.54,0.29,2.0,0.17,562,3,2400
7,0.147610,0.54,0.30,2.0,0.18,568,3,2400
8,0.147610,0.54,0.30,2.0,0.17,562,3,2400
9,0.147610,0.54,0.30,2.0,0.15,556,3,2400


최적 OOF MAE: 0.1476033333333333


## 6. 제출 후보 선택

서로 거의 같은 후보만 저장하지 않고 다음 성격으로 나눕니다.

- `cv_best`: CV 최적
- `safe`: 기존 설정에 가까운 후보
- `tree_heavy`: Pair 비중이 낮은 후보
- `pair_heavy`: Pair 비중이 높은 후보
- `no_override`: Override 없는 후보


In [7]:
def row_to_config(
    row: pd.Series,
) -> dict:
    return {
        "tree_quantile": float(
            row["tree_quantile"]
        ),
        "pair_weight": float(
            row["pair_weight"]
        ),
        "round_decimals": (
            None
            if pd.isna(
                row["round_decimals"]
            )
            else int(
                row["round_decimals"]
            )
        ),
        "override_threshold": (
            None
            if pd.isna(
                row["override_threshold"]
            )
            else float(
                row["override_threshold"]
            )
        ),
        "cv_mae": float(
            row["cv_mae"]
        ),
    }


candidate_rows = [
    ("cv_best", search_df.iloc[0]),
]

safe_rows = search_df[
    (search_df["tree_quantile"].between(
        0.53,
        0.54,
    ))
    & (search_df["pair_weight"].between(
        0.27,
        0.29,
    ))
    & (search_df["round_decimals"] == 2)
    & (
        search_df["override_threshold"]
        .isin([0.18, 0.19, 0.20])
    )
]

if len(safe_rows):
    candidate_rows.append(
        ("safe", safe_rows.iloc[0])
    )

tree_heavy_rows = search_df[
    search_df["pair_weight"]
    == min(PAIR_WEIGHTS)
]

if len(tree_heavy_rows):
    candidate_rows.append(
        (
            "tree_heavy",
            tree_heavy_rows.iloc[0],
        )
    )

pair_heavy_rows = search_df[
    search_df["pair_weight"]
    == max(PAIR_WEIGHTS)
]

if len(pair_heavy_rows):
    candidate_rows.append(
        (
            "pair_heavy",
            pair_heavy_rows.iloc[0],
        )
    )

no_override_rows = search_df[
    search_df["override_threshold"].isna()
]

if len(no_override_rows):
    candidate_rows.append(
        (
            "no_override",
            no_override_rows.iloc[0],
        )
    )

FINAL_CONFIGS = []
seen = set()

for name, row in candidate_rows:
    config = row_to_config(row)

    key = (
        config["tree_quantile"],
        config["pair_weight"],
        config["round_decimals"],
        config["override_threshold"],
    )

    if key not in seen:
        seen.add(key)
        FINAL_CONFIGS.append(
            (name, config)
        )

FINAL_CONFIGS = FINAL_CONFIGS[:5]

print("최종 제출 후보")
for name, config in FINAL_CONFIGS:
    print(name, config)


최종 제출 후보
cv_best {'tree_quantile': 0.54, 'pair_weight': 0.29, 'round_decimals': 2, 'override_threshold': 0.16, 'cv_mae': 0.1476033333333333}
safe {'tree_quantile': 0.54, 'pair_weight': 0.29, 'round_decimals': 2, 'override_threshold': 0.19, 'cv_mae': 0.1476033333333333}
tree_heavy {'tree_quantile': 0.54, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14762666666666666}
pair_heavy {'tree_quantile': 0.54, 'pair_weight': 0.31, 'round_decimals': None, 'override_threshold': 0.2, 'cv_mae': 0.14764805066666667}
no_override {'tree_quantile': 0.54, 'pair_weight': 0.29, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14777666666666667}


## 7. 전체 데이터 학습 및 제출 파일 생성


In [8]:
if not sample_path.exists():
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(
        sample_path
    )

bounds = compute_iqr_bounds(
    raw_input,
    WINSORIZE_COLUMNS,
)

train_winsorized = apply_winsorize(
    raw_input,
    bounds,
)

test_winsorized = apply_winsorize(
    test_input,
    bounds,
)

train_feat = add_features(
    train_winsorized
)

test_feat = add_features(
    test_winsorized
)

test_seed_matrices = []

for seed in SEEDS:
    seed_matrix = extra_trees_seed_matrix(
        train_feat,
        test_feat,
        y,
        seed,
    )

    test_seed_matrices.append(
        seed_matrix
    )

    print(
        f"전체 데이터 seed {seed} 완료"
    )

combined_test_tree_matrix = np.concatenate(
    test_seed_matrices,
    axis=1,
)

test_pair_pred = pair_prediction(
    train_feat,
    test_feat,
    y,
)

test_nearest_dist, test_nearest_target = (
    nearest_neighbor_lookup(
        raw_input,
        test_input,
        y,
    )
)

saved_paths = []

for rank, (name, config) in enumerate(
    FINAL_CONFIGS,
    start=1,
):
    tree_pred = np.quantile(
        combined_test_tree_matrix,
        config["tree_quantile"],
        axis=1,
    )

    blended = (
        (1.0 - config["pair_weight"])
        * tree_pred
        + config["pair_weight"]
        * test_pair_pred
    )

    final_pred = apply_postprocess(
        blended,
        config["round_decimals"],
        config["override_threshold"],
        test_nearest_dist,
        test_nearest_target,
    )

    submission = submission_template.copy()
    submission[TARGET] = final_pred

    filename = (
        f"submit_multiseed_v10_rank{rank}_{name}"
        f"_tq{config['tree_quantile']:.3f}"
        f"_w{config['pair_weight']:.2f}"
        f"_th{config['override_threshold']}.csv"
    )

    output_file = OUTPUT_DIR / filename

    submission.to_csv(
        output_file,
        index=False,
    )

    saved_paths.append(
        output_file
    )

    print("\nSaved:", output_file)
    print("Config:", config)
    print(
        "Prediction min/mean/max:",
        float(final_pred.min()),
        float(final_pred.mean()),
        float(final_pred.max()),
    )
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)


전체 데이터 seed 42 완료
전체 데이터 seed 2024 완료
전체 데이터 seed 777 완료

Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank1_cv_best_tq0.540_w0.29_th0.16.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.29, 'round_decimals': 2, 'override_threshold': 0.16, 'cv_mae': 0.1476033333333333}
Prediction min/mean/max: 0.0 0.50982 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank2_safe_tq0.540_w0.29_th0.19.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.29, 'round_decimals': 2, 'override_threshold': 0.19, 'cv_mae': 0.1476033333333333}
Prediction min/mean/max: 0.0 0.50982 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank3_tree_heavy_tq0.540_w0.25_th0.2.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14762666666666666}
Prediction min/mean/max: 0.0 0.5107666666666667 1.0


...


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank4_pair_heavy_tq0.540_w0.31_th0.2.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.31, 'round_decimals': None, 'override_threshold': 0.2, 'cv_mae': 0.14764805066666667}
Prediction min/mean/max: 0.0 0.5092322633333333 1.0


,ID,stress_score
0,TEST_0000,0.617556
1,TEST_0001,0.970000
2,TEST_0002,0.190000
3,TEST_0003,0.525376
4,TEST_0004,0.530000



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank5_no_override_tq0.540_w0.29_thNone.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.29, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14777666666666667}
Prediction min/mean/max: 0.0 0.5098766666666668 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank1_cv_best_tq0.540_w0.29_th0.16.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank2_safe_tq0.540_w0.29_th0.19.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank3_tree_heavy_tq0.540_w0.25_th0.2.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank4_pair_heavy_tq0.540_w0.31_th0.2.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_v10_rank5_no_override_tq0.540_w0.29_thNone.csv


## 확인할 결과

1. `seed_check_df`
   - Multi-Seed ExtraTrees 단독으로 안정화됐는지 확인
2. `search_df.head(40)`
   - 기존 `0.147747`보다 낮아졌는지 확인
3. 최종 후보
   - 제출 제한이 있다면 우선 `cv_best`, 다음 `safe` 순서 추천
